GET RESULTS

In [1]:
from experiment_main import *

from math import floor
import csv
from os import listdir  # , remove

# sns.set()
import time

macd_params = (12, 26, 9)
# mode = "1_day"
alpha_range = []
gamma_range = []
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)
gen = [(x, y) for x in alpha_range for y in gamma_range]


def update_bh_results_dict(results_dict, current_period_bh, experiment, time_format):
    results_dict[experiment][time_format]["bets"] += 1
    results_dict[experiment][time_format]["return"] += current_period_bh
    if current_period_bh > 0:
        results_dict[experiment][time_format]["wins"] += 1
        results_dict[experiment][time_format]["profit"] += current_period_bh

In [2]:
# market = 'soybean'
market = "stocks"
trade_cost = False
time_formats = ["1_day", "1_hour", "5_minute"]
experiments = [0, 20000]
data_files = [
    "aapl.us.txt",
    "amgn.us.txt",
    "amzn.us.txt",
    "axp.us.txt",
    "ba.us.txt",
    "cat.us.txt",
    "crm.us.txt",
    "csco.us.txt",
    "cvx.us.txt",
    "dis.us.txt",
    "fb.us.txt",
    "googl.us.txt",
    "gs.us.txt",
    "hd.us.txt",
    "hon.us.txt",
    "ibm.us.txt",
    "jnj.us.txt",
    "jpm.us.txt",
    "ko.us.txt",
    "mcd.us.txt",
    "meta.us.txt",
    "mmm.us.txt",
    "mrk.us.txt",
    "msft.us.txt",
    "nflx.us.txt",
    "nke.us.txt",
    "nvda.us.txt",
    "pg.us.txt",
    "shw.us.txt",
    "trv.us.txt",
    "unh.us.txt",
    "v.us.txt",
    "vz.us.txt",
    "wmt.us.txt",
]

# if time_format == "1_day":
#     data_files = listdir(prefix)
# else:
#     data_files = listdir(prefix + '1/')
data_files.remove("meta.us.txt")
data_files.remove("fb.us.txt")

sorted_ticks = sorted(data_files)
data_files

['aapl.us.txt',
 'amgn.us.txt',
 'amzn.us.txt',
 'axp.us.txt',
 'ba.us.txt',
 'cat.us.txt',
 'crm.us.txt',
 'csco.us.txt',
 'cvx.us.txt',
 'dis.us.txt',
 'googl.us.txt',
 'gs.us.txt',
 'hd.us.txt',
 'hon.us.txt',
 'ibm.us.txt',
 'jnj.us.txt',
 'jpm.us.txt',
 'ko.us.txt',
 'mcd.us.txt',
 'mmm.us.txt',
 'mrk.us.txt',
 'msft.us.txt',
 'nflx.us.txt',
 'nke.us.txt',
 'nvda.us.txt',
 'pg.us.txt',
 'shw.us.txt',
 'trv.us.txt',
 'unh.us.txt',
 'v.us.txt',
 'vz.us.txt',
 'wmt.us.txt']

Stocks Under Study Table

In [3]:
outer_keys = [
    "Stock",
    "Ticker",
    "Capitalization",
    "1_day Gain",
    "1_hour Gain",
    "5_minute Gain",
]
stocks_under_study_dict = {x: [] for x in outer_keys}
stocks_under_study_dict["Stock"] = [ticker for ticker in sorted_ticks]
stocks_under_study_dict["Ticker"] = [ticker[:-7].upper() for ticker in sorted_ticks]
stocks_under_study_dict["Capitalization"] = [ticker for ticker in sorted_ticks]

for time_format, gain in zip(
    time_formats, ["1_day Gain", "1_hour Gain", "5_minute Gain"]
):
    # time_format_formated = f"{time_format.split("_")[0]} {time_format.split("_")[1].capitalize()}"
    prefix = f"./data/stocks/{time_format}/"
    isMonth = "month"
    if time_format == "5_minute":  # 5_minute
        isMonth = "week"
    for ticker in sorted_ticks:
        # formated_ticker = ticker[:-7].upper()
        # precent increase
        df_read = read_and_change_data(prefix, ticker)
        percent_increase = (
            (df_read.iloc[-1]["close"] - df_read.iloc[0]["close"])
            / df_read.iloc[0]["close"]
        ) * 100

        stocks_under_study_dict[gain].append(f"{round(percent_increase, 2)}%")

for x in stocks_under_study_dict:
    print(x, len(stocks_under_study_dict[x]))


stocks_under_study_df = pd.DataFrame(stocks_under_study_dict)
stocks_under_study_df.to_csv("./results/buy_and_hold/stocks_under_study_df", index=None)
stocks_under_study_df

Stock 32
Ticker 32
Capitalization 32
1_day Gain 32
1_hour Gain 32
5_minute Gain 32


,Stock,Ticker,Capitalization,1_day Gain,1_hour Gain,5_minute Gain
0,aapl.us.txt,AAPL,aapl.us.txt,187.6%,184.19%,183.66%
1,amgn.us.txt,AMGN,amgn.us.txt,33.05%,33.0%,31.58%
2,amzn.us.txt,AMZN,amzn.us.txt,128.09%,132.23%,131.88%
3,axp.us.txt,AXP,axp.us.txt,141.42%,141.09%,140.46%
4,ba.us.txt,BA,ba.us.txt,-46.76%,-47.13%,-47.58%
5,cat.us.txt,CAT,cat.us.txt,169.36%,162.04%,161.96%
6,crm.us.txt,CRM,crm.us.txt,85.23%,84.95%,84.96%
7,csco.us.txt,CSCO,csco.us.txt,29.96%,28.33%,28.32%
8,cvx.us.txt,CVX,cvx.us.txt,40.54%,40.32%,40.33%
9,dis.us.txt,DIS,dis.us.txt,-20.89%,-21.3%,-21.29%


In [4]:
# results for every experiment and every time format and every ticker and every label (timespan)
buy_and_hold_dict = {
    exkey: {
        key: {t: dict() for t in ["total"] + data_files}
        for key in ["1_day", "1_hour", "5_minute"]
    }
    for exkey in [0, 20000]
}

# list of optimization and validation periods for every time format and every experiment
periods_dict = {
    ee: {tt: {"test": [], "valid": []} for tt in ["1_day", "1_hour", "5_minute"]}
    for ee in [0, 20000]
}

# overall results for every experiment and every time format
current_results_dict = {
    ee: {
        tt: {"bets": 0, "wins": 0, "return": 0, "profit": 0}
        for tt in ["1_day", "1_hour", "5_minute"]
    }
    for ee in [0, 20000]
}
buy_and_hold_dict
# periods_dict
# current_results_dict

{0: {'1_day': {'total': {},
   'aapl.us.txt': {},
   'amgn.us.txt': {},
   'amzn.us.txt': {},
   'axp.us.txt': {},
   'ba.us.txt': {},
   'cat.us.txt': {},
   'crm.us.txt': {},
   'csco.us.txt': {},
   'cvx.us.txt': {},
   'dis.us.txt': {},
   'googl.us.txt': {},
   'gs.us.txt': {},
   'hd.us.txt': {},
   'hon.us.txt': {},
   'ibm.us.txt': {},
   'jnj.us.txt': {},
   'jpm.us.txt': {},
   'ko.us.txt': {},
   'mcd.us.txt': {},
   'mmm.us.txt': {},
   'mrk.us.txt': {},
   'msft.us.txt': {},
   'nflx.us.txt': {},
   'nke.us.txt': {},
   'nvda.us.txt': {},
   'pg.us.txt': {},
   'shw.us.txt': {},
   'trv.us.txt': {},
   'unh.us.txt': {},
   'v.us.txt': {},
   'vz.us.txt': {},
   'wmt.us.txt': {}},
  '1_hour': {'total': {},
   'aapl.us.txt': {},
   'amgn.us.txt': {},
   'amzn.us.txt': {},
   'axp.us.txt': {},
   'ba.us.txt': {},
   'cat.us.txt': {},
   'crm.us.txt': {},
   'csco.us.txt': {},
   'cvx.us.txt': {},
   'dis.us.txt': {},
   'googl.us.txt': {},
   'gs.us.txt': {},
   'hd.us.txt': 

In [5]:
# buy_and_hold_dict
periods_dict
# current_results_dict

{0: {'1_day': {'test': [], 'valid': []},
  '1_hour': {'test': [], 'valid': []},
  '5_minute': {'test': [], 'valid': []}},
 20000: {'1_day': {'test': [], 'valid': []},
  '1_hour': {'test': [], 'valid': []},
  '5_minute': {'test': [], 'valid': []}}}

In [6]:
# buy_and_hold_dict
# periods_dict
current_results_dict

{0: {'1_day': {'bets': 0, 'wins': 0, 'return': 0, 'profit': 0},
  '1_hour': {'bets': 0, 'wins': 0, 'return': 0, 'profit': 0},
  '5_minute': {'bets': 0, 'wins': 0, 'return': 0, 'profit': 0}},
 20000: {'1_day': {'bets': 0, 'wins': 0, 'return': 0, 'profit': 0},
  '1_hour': {'bets': 0, 'wins': 0, 'return': 0, 'profit': 0},
  '5_minute': {'bets': 0, 'wins': 0, 'return': 0, 'profit': 0}}}

In [7]:
starting_time = time.time()
for exp in experiments:
    for time_format in ["1_day", "1_hour"]:
        prefix = f"./data/stocks/{time_format}/"
        isMonth = "month"
        if time_format == "1_hour":
            time_value = 3
        else:
            time_value = 12

        for index, tick in enumerate(data_files):
            print(f"current tick: {tick}")

            df = make_dataframe(
                prefix, tick, time_format, macd_params, alpha_range, gamma_range
            )

            start = df.iloc[0].name
            end = df.iloc[-1].name
            test_period_start = start
            test_period_end = make_new_time(test_period_start, time_value, isMonth)
            validation_period_end = make_new_time(test_period_end, time_value, isMonth)

            print(
                f"before loop start: {start}, end: {end}, test_period_start: {test_period_start}, test_period_end: {test_period_end}, validation_period_end: {validation_period_end}"
            )

            period_counter = 0
            current_tick_bh = 0
            # current_tick_results_dict = {"bets": 0, "wins": 0, "return": 0, "profit": 0}

            while validation_period_end <= end:
                period_counter += 1

                df_test = df[test_period_start:test_period_end]
                df_valid = df[test_period_end:validation_period_end]
                # print(isMonth, test_period_start, test_period_end, test_period_end - test_period_start)
                # print(test_period_end, validation_period_end, validation_period_end - test_period_end)
                # if df_test.isnull().values.any() or df_valid.isnull().values.any():
                #     raise ValueError
                # test_period_end:validation_period_end
                label = get_time_out(
                    f"{test_period_end.year}-{test_period_end.month}-{test_period_end.day}:{validation_period_end.year}-{validation_period_end.month}-{validation_period_end.day}"
                )

                # test_period_start:test_period_end
                periods_dict[exp][time_format]["test"].append(
                    get_time_out(
                        f"{test_period_start.year}-{test_period_start.month}-{test_period_start.day}:{test_period_end.year}-{test_period_end.month}-{test_period_end.day}"
                    )
                )
                periods_dict[exp][time_format]["test"].append(
                    f"{test_period_start}:{test_period_end}"
                )

                periods_dict[exp][time_format]["valid"].append(label)
                periods_dict[exp][time_format]["valid"].append(
                    f"{test_period_end}:{validation_period_end}"
                )

                strategy = "Buy and Hold"
                print(f"start of {strategy}")

                period_bh = buy_and_hold(df_valid, exp)
                update_bh_results_dict(
                    current_results_dict, period_bh, exp, time_format
                )

                buy_and_hold_dict[exp][time_format][tick][label] = period_bh
                buy_and_hold_dict[exp][time_format]["total"][label] = (
                    buy_and_hold_dict[exp][time_format]["total"].get(label, 0)
                    + period_bh
                )
                current_tick_bh += period_bh

                print(
                    f"tick: {tick}, tick index: {data_files.index(tick) + 1}, market: {market}, time_format: {time_format}"
                )
                print("--- %s seconds ---" % (time.time() - starting_time))
                # same for all time formats

                if time_format == "1_day":
                    test_period_start = make_new_time(test_period_start, 3, isMonth)
                    test_period_end = make_new_time(test_period_end, 3, isMonth)
                    validation_period_end = make_new_time(
                        validation_period_end, 3, isMonth
                    )
                else:  # elif time_format == "1_hour":
                    test_period_start = make_new_time(test_period_start, 2, "week")
                    test_period_end = make_new_time(test_period_end, 2, "week")
                    validation_period_end = make_new_time(
                        validation_period_end, 2, "week"
                    )

                print(
                    f"end of loop exp: {exp}, time_format: {time_format} start: {start}, end: {end}, test_period_start: {test_period_start}, test_period_end: {test_period_end}, validation_period_end: {validation_period_end}"
                )

            buy_and_hold_dict[exp][time_format][tick]["average_per_time"] = (
                current_tick_bh / period_counter
            )

            buy_and_hold_dict[exp][time_format][tick]["all"] = current_tick_bh
            buy_and_hold_dict[exp][time_format]["total"]["all"] = (
                buy_and_hold_dict[exp][time_format]["total"].get("all", 0)
                + current_tick_bh
            )

            print(
                f"out of loop start: {start}, end: {end}, test_period_start: {test_period_start}, test_period_end: {test_period_end}, validation_period_end: {validation_period_end}"
            )

        label_length = len(periods_dict[exp][time_format]["valid"]) / 2
        buy_and_hold_dict[exp][time_format]["total"]["average_per_ticker"] = (
            buy_and_hold_dict[exp][time_format]["total"]["all"] / len(data_files)
        )
        buy_and_hold_dict[exp][time_format]["total"]["average_per_time"] = (
            buy_and_hold_dict[exp][time_format]["total"]["all"] / label_length
        )

        current_results_dict[exp][time_format]["losses"] = (
            current_results_dict[exp][time_format]["bets"]
            - current_results_dict[exp][time_format]["wins"]
        )
        current_results_dict[exp][time_format]["loss"] = (
            current_results_dict[exp][time_format]["return"]
            - current_results_dict[exp][time_format]["profit"]
        )

        current_results_dict[exp][time_format]["average_return_per_ticker"] = (
            current_results_dict[exp][time_format]["return"] / len(data_files)
        )
        current_results_dict[exp][time_format]["average_profit_per_ticker"] = (
            current_results_dict[exp][time_format]["profit"] / len(data_files)
        )
        current_results_dict[exp][time_format]["average_loss_per_ticker"] = (
            current_results_dict[exp][time_format]["loss"] / len(data_files)
        )

        current_results_dict[exp][time_format]["average_return_per_time"] = (
            current_results_dict[exp][time_format]["return"] / label_length
        )
        current_results_dict[exp][time_format]["average_profit_per_time"] = (
            current_results_dict[exp][time_format]["profit"] / label_length
        )
        current_results_dict[exp][time_format]["average_loss_per_time"] = (
            current_results_dict[exp][time_format]["loss"] / label_length
        )

current tick: aapl.us.txt
before loop start: 2020-03-26 04:00:00, end: 2025-02-05 05:00:00, test_period_start: 2020-03-26 04:00:00, test_period_end: 2021-03-26 00:00:00, validation_period_end: 2022-03-26 00:00:00
start of Buy and Hold
tick: aapl.us.txt, tick index: 1, market: stocks, time_format: 1_day
--- 0.02898883819580078 seconds ---
end of loop exp: 0, time_format: 1_day start: 2020-03-26 04:00:00, end: 2025-02-05 05:00:00, test_period_start: 2020-06-26 00:00:00, test_period_end: 2021-06-26 00:00:00, validation_period_end: 2022-06-26 00:00:00
start of Buy and Hold
tick: aapl.us.txt, tick index: 1, market: stocks, time_format: 1_day
--- 0.029893875122070312 seconds ---
end of loop exp: 0, time_format: 1_day start: 2020-03-26 04:00:00, end: 2025-02-05 05:00:00, test_period_start: 2020-09-26 00:00:00, test_period_end: 2021-09-26 00:00:00, validation_period_end: 2022-09-26 00:00:00
start of Buy and Hold
tick: aapl.us.txt, tick index: 1, market: stocks, time_format: 1_day
--- 0.0307791

In [8]:
starting_time = time.time()
for exp in experiments:
    for time_format in ["5_minute"]:
        prefix = f"./data/stocks/{time_format}/"
        isMonth = "week"
        time_value = 8

        for index, tick in enumerate(data_files):
            print(f"current tick: {tick}")

            df_read = read_and_change_data(prefix, tick)
            index_list = list(df_read.index)

            start = df_read.iloc[0].name
            end = df_read.iloc[-1].name

            nan_indexes = (max(macd_params[0], macd_params[1]) + macd_params[2]) - 2

            test_period_start = df_read.iloc[nan_indexes].name
            test_period_end = make_new_time(test_period_start, time_value, isMonth)
            validation_period_end = make_new_time(test_period_end, time_value, isMonth)

            print(
                f"before loop start: {start}, end: {end}, test_period_start: {test_period_start}, test_period_end: {test_period_end}, validation_period_end: {validation_period_end}"
            )

            period_counter = 0
            current_tick_bh = 0
            # current_tick_results_dict = {"bets": 0, "wins": 0, "return": 0, "profit": 0}

            while validation_period_end <= end:
                period_counter += 1

                # test_period_start_index = index_list.index(test_period_start)
                test_period_start_index = get_index(index_list, test_period_start)
                calculate_period_start = df_read.iloc[
                    max(test_period_start_index - nan_indexes - 500, 0)
                ].name
                calculate_period_end = validation_period_end

                # print(isMonth, test_period_start_index, max(test_period_start_index - nan_indexes - 500, 0))
                # print(test_period_start, test_period_end, test_period_end - test_period_start)
                # print(test_period_end, validation_period_end, validation_period_end - test_period_end)
                # print(calculate_period_start, calculate_period_end, calculate_period_end - calculate_period_start)

                df_calc = calculate_indicators(
                    df_read[calculate_period_start:calculate_period_end],
                    col="close",
                    ema_mode="normal",
                    short_period=macd_params[0],
                    long_period=macd_params[1],
                    signal_period=macd_params[2],
                    alpha_range=alpha_range,
                    gamma_range=gamma_range,
                )
                df_test = df_calc[test_period_start:test_period_end]
                df_valid = df_calc[test_period_end:validation_period_end]
                # df_test = df_read[test_period_start:test_period_end]
                # df_valid = df_read[test_period_end:validation_period_end]
                # test_period_end:validation_period_end
                label = get_time_out(
                    f"{test_period_end.year}-{test_period_end.month}-{test_period_end.day}:{validation_period_end.year}-{validation_period_end.month}-{validation_period_end.day}"
                )

                # test_period_start:test_period_end
                periods_dict[exp][time_format]["test"].append(
                    get_time_out(
                        f"{test_period_start.year}-{test_period_start.month}-{test_period_start.day}:{test_period_end.year}-{test_period_end.month}-{test_period_end.day}"
                    )
                )
                periods_dict[exp][time_format]["test"].append(
                    f"{test_period_start}:{test_period_end}"
                )

                periods_dict[exp][time_format]["valid"].append(label)
                periods_dict[exp][time_format]["valid"].append(
                    f"{test_period_end}:{validation_period_end}"
                )

                strategy = "Buy and Hold"
                print(f"start of {strategy}")

                period_bh = buy_and_hold(df_valid, exp)
                update_bh_results_dict(
                    current_results_dict, period_bh, exp, time_format
                )

                buy_and_hold_dict[exp][time_format][tick][label] = period_bh
                buy_and_hold_dict[exp][time_format]["total"][label] = (
                    buy_and_hold_dict[exp][time_format]["total"].get(label, 0)
                    + period_bh
                )
                current_tick_bh += period_bh

                print(
                    f"tick: {tick}, tick index: {data_files.index(tick) + 1}, market: {market}, time_format: {time_format}"
                )
                print("--- %s seconds ---" % (time.time() - starting_time))
                # same for all time formats

                test_period_start = make_new_time(test_period_start, 1, isMonth)
                test_period_end = make_new_time(test_period_end, 1, isMonth)
                validation_period_end = make_new_time(validation_period_end, 1, isMonth)

                print(
                    f"end of loop exp: {exp}, time_format: {time_format} start: {start}, end: {end}, test_period_start: {test_period_start}, test_period_end: {test_period_end}, validation_period_end: {validation_period_end}"
                )

            buy_and_hold_dict[exp][time_format][tick]["average_per_time"] = (
                current_tick_bh / period_counter
            )

            buy_and_hold_dict[exp][time_format][tick]["all"] = current_tick_bh
            buy_and_hold_dict[exp][time_format]["total"]["all"] = (
                buy_and_hold_dict[exp][time_format]["total"].get("all", 0)
                + current_tick_bh
            )

            print(
                f"out of loop start: {start}, end: {end}, test_period_start: {test_period_start}, test_period_end: {test_period_end}, validation_period_end: {validation_period_end}"
            )

        label_length = len(periods_dict[exp][time_format]["valid"]) / 2
        buy_and_hold_dict[exp][time_format]["total"]["average_per_ticker"] = (
            buy_and_hold_dict[exp][time_format]["total"]["all"] / len(data_files)
        )
        buy_and_hold_dict[exp][time_format]["total"]["average_per_time"] = (
            buy_and_hold_dict[exp][time_format]["total"]["all"] / label_length
        )

        current_results_dict[exp][time_format]["losses"] = (
            current_results_dict[exp][time_format]["bets"]
            - current_results_dict[exp][time_format]["wins"]
        )
        current_results_dict[exp][time_format]["loss"] = (
            current_results_dict[exp][time_format]["return"]
            - current_results_dict[exp][time_format]["profit"]
        )

        current_results_dict[exp][time_format]["average_return_per_ticker"] = (
            current_results_dict[exp][time_format]["return"] / len(data_files)
        )
        current_results_dict[exp][time_format]["average_profit_per_ticker"] = (
            current_results_dict[exp][time_format]["profit"] / len(data_files)
        )
        current_results_dict[exp][time_format]["average_loss_per_ticker"] = (
            current_results_dict[exp][time_format]["loss"] / len(data_files)
        )

        current_results_dict[exp][time_format]["average_return_per_time"] = (
            current_results_dict[exp][time_format]["return"] / label_length
        )
        current_results_dict[exp][time_format]["average_profit_per_time"] = (
            current_results_dict[exp][time_format]["profit"] / label_length
        )
        current_results_dict[exp][time_format]["average_loss_per_time"] = (
            current_results_dict[exp][time_format]["loss"] / label_length
        )

current tick: aapl.us.txt
before loop start: 2020-02-07 09:00:00, end: 2025-02-05 17:50:00, test_period_start: 2020-02-07 12:20:00, test_period_end: 2020-04-03 00:00:00, validation_period_end: 2020-05-29 00:00:00
start of Buy and Hold
tick: aapl.us.txt, tick index: 1, market: stocks, time_format: 5_minute
--- 2.409392833709717 seconds ---
end of loop exp: 0, time_format: 5_minute start: 2020-02-07 09:00:00, end: 2025-02-05 17:50:00, test_period_start: 2020-02-14 00:00:00, test_period_end: 2020-04-10 00:00:00, validation_period_end: 2020-06-05 00:00:00
start of Buy and Hold
tick: aapl.us.txt, tick index: 1, market: stocks, time_format: 5_minute
--- 3.4201931953430176 seconds ---
end of loop exp: 0, time_format: 5_minute start: 2020-02-07 09:00:00, end: 2025-02-05 17:50:00, test_period_start: 2020-02-21 00:00:00, test_period_end: 2020-04-17 00:00:00, validation_period_end: 2020-06-12 00:00:00
start of Buy and Hold
tick: aapl.us.txt, tick index: 1, market: stocks, time_format: 5_minute
--

In [9]:
periods_dict

{0: {'1_day': {'test': ['2020-03-26:2021-03-26',
    '2020-03-26 04:00:00:2021-03-26 00:00:00',
    '2020-06-26:2021-06-26',
    '2020-06-26 00:00:00:2021-06-26 00:00:00',
    '2020-09-26:2021-09-26',
    '2020-09-26 00:00:00:2021-09-26 00:00:00',
    '2020-12-26:2021-12-26',
    '2020-12-26 00:00:00:2021-12-26 00:00:00',
    '2021-03-26:2022-03-26',
    '2021-03-26 00:00:00:2022-03-26 00:00:00',
    '2021-06-26:2022-06-26',
    '2021-06-26 00:00:00:2022-06-26 00:00:00',
    '2021-09-26:2022-09-26',
    '2021-09-26 00:00:00:2022-09-26 00:00:00',
    '2021-12-26:2022-12-26',
    '2021-12-26 00:00:00:2022-12-26 00:00:00',
    '2022-03-26:2023-03-26',
    '2022-03-26 00:00:00:2023-03-26 00:00:00',
    '2022-06-26:2023-06-26',
    '2022-06-26 00:00:00:2023-06-26 00:00:00',
    '2022-09-26:2023-09-26',
    '2022-09-26 00:00:00:2023-09-26 00:00:00',
    '2022-12-26:2023-12-26',
    '2022-12-26 00:00:00:2023-12-26 00:00:00',
    '2020-03-26:2021-03-26',
    '2020-03-26 04:00:00:2021-03-26 00:

In [10]:
buy_and_hold_dict

{0: {'1_day': {'total': {'2021-03-26:2022-03-26': 213.19510000000002,
    '2021-06-26:2022-06-26': -808.8200000000003,
    '2021-09-26:2022-09-26': -1272.671,
    '2021-12-26:2022-12-26': -999.7677,
    '2022-03-26:2023-03-26': -550.4957000000002,
    '2022-06-26:2023-06-26': 651.6904,
    '2022-09-26:2023-09-26': 985.2449999999999,
    '2022-12-26:2023-12-26': 1042.2653,
    '2023-03-26:2024-03-26': 1636.8853000000001,
    '2023-06-26:2024-06-26': 1314.1199000000001,
    '2023-09-26:2024-09-26': 1890.1422999999995,
    '2023-12-26:2024-12-26': 1582.0512999999999,
    'all': 5683.840199999999,
    'average_per_ticker': 177.62000624999996,
    'average_per_time': 14.801667187499996},
   'aapl.us.txt': {'2021-03-26:2022-03-26': 53.510000000000005,
    '2021-06-26:2022-06-26': 6.8799999999999955,
    '2021-09-26:2022-09-26': 5.060000000000002,
    '2021-12-26:2022-12-26': -48.47,
    '2022-03-26:2023-03-26': -15.349999999999994,
    '2022-06-26:2023-06-26': 45.02000000000001,
    '2022-09

In [12]:
current_results_dict

{0: {'1_day': {'bets': 384,
   'wins': 232,
   'return': 5683.840199999997,
   'profit': 11084.886599999998,
   'losses': 152,
   'loss': -5401.046400000001,
   'average_return_per_ticker': 177.6200062499999,
   'average_profit_per_ticker': 346.40270624999994,
   'average_loss_per_ticker': -168.78270000000003,
   'average_return_per_time': 14.801667187499993,
   'average_profit_per_time': 28.866892187499996,
   'average_loss_per_time': -14.065225000000003},
  '1_hour': {'bets': 3775,
   'wins': 2290,
   'return': 23763.326499999985,
   'profit': 48028.23940000015,
   'losses': 1485,
   'loss': -24264.912900000167,
   'average_return_per_ticker': 742.6039531249995,
   'average_profit_per_ticker': 1500.8824812500047,
   'average_loss_per_ticker': -758.2785281250052,
   'average_return_per_time': 6.294920927152313,
   'average_profit_per_time': 12.7227124238411,
   'average_loss_per_time': -6.427791496688786},
  '5_minute': {'bets': 7840,
   'wins': 4616,
   'return': 31635.095377000038,


In [ ]:
def make_paired_dict(optimization_periods, validation_periods):
    paired_dict = dict()
    # for i in range(0, len(optimization_periods)):
    for i in range(len(optimization_periods)):
        # print(i)
        paired_dict[optimization_periods[i]] = validation_periods[i], i
        paired_dict[validation_periods[i]] = optimization_periods[i], i
    return paired_dict


for time_format in time_formats:
    for exp in experiments:
        optimization_periods = periods_dict[exp][time_format]["test"]  # optimization
        write_json_to_file(
            f"./results/buy_and_hold/",
            f"optimization_periods_{time_format}_{exp}",
            optimization_periods,
        )
        # len(optimization_periods)

        validation_periods = periods_dict[exp][time_format]["valid"]  # validation
        write_json_to_file(
            f"./results/buy_and_hold/",
            f"validation_periods_{time_format}_{exp}",
            validation_periods,
        )
        # len(validation_periods)

        paired_dict = make_paired_dict(optimization_periods, validation_periods)
        write_json_to_file(
            f"./results/buy_and_hold/", f"paired_dict_{time_format}_{exp}", paired_dict
        )

In [14]:
write_json_to_file(f"./results/buy_and_hold/", "bh_result", buy_and_hold_dict)
write_json_to_file(f"./results/buy_and_hold/", "bh_result_table", current_results_dict)
write_json_to_file(f"./results/buy_and_hold/", "periods_dict", periods_dict)

------------------------------

GET TABLES

In [1]:
from experiment_main import *

from math import floor
import csv
from os import listdir  # , remove

# sns.set()
import time

macd_params = (12, 26, 9)
# mode = "1_day"
alpha_range = []
gamma_range = []
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)
gen = [(x, y) for x in alpha_range for y in gamma_range]


def update_bh_results_dict(results_dict, current_period_bh, experiment, time_format):
    results_dict[experiment][time_format]["bets"] += 1
    results_dict[experiment][time_format]["return"] += current_period_bh
    if current_period_bh > 0:
        results_dict[experiment][time_format]["wins"] += 1
        results_dict[experiment][time_format]["profit"] += current_period_bh


# market = 'soybean'
market = "stocks"
trade_cost = False
time_formats = ["1_day", "1_hour", "5_minute"]
experiments = [0, 20000]
data_files = [
    "aapl.us.txt",
    "amgn.us.txt",
    "amzn.us.txt",
    "axp.us.txt",
    "ba.us.txt",
    "cat.us.txt",
    "crm.us.txt",
    "csco.us.txt",
    "cvx.us.txt",
    "dis.us.txt",
    "fb.us.txt",
    "googl.us.txt",
    "gs.us.txt",
    "hd.us.txt",
    "hon.us.txt",
    "ibm.us.txt",
    "jnj.us.txt",
    "jpm.us.txt",
    "ko.us.txt",
    "mcd.us.txt",
    "meta.us.txt",
    "mmm.us.txt",
    "mrk.us.txt",
    "msft.us.txt",
    "nflx.us.txt",
    "nke.us.txt",
    "nvda.us.txt",
    "pg.us.txt",
    "shw.us.txt",
    "trv.us.txt",
    "unh.us.txt",
    "v.us.txt",
    "vz.us.txt",
    "wmt.us.txt",
]

data_files.remove("meta.us.txt")
data_files.remove("fb.us.txt")
data_files

['aapl.us.txt',
 'amgn.us.txt',
 'amzn.us.txt',
 'axp.us.txt',
 'ba.us.txt',
 'cat.us.txt',
 'crm.us.txt',
 'csco.us.txt',
 'cvx.us.txt',
 'dis.us.txt',
 'googl.us.txt',
 'gs.us.txt',
 'hd.us.txt',
 'hon.us.txt',
 'ibm.us.txt',
 'jnj.us.txt',
 'jpm.us.txt',
 'ko.us.txt',
 'mcd.us.txt',
 'mmm.us.txt',
 'mrk.us.txt',
 'msft.us.txt',
 'nflx.us.txt',
 'nke.us.txt',
 'nvda.us.txt',
 'pg.us.txt',
 'shw.us.txt',
 'trv.us.txt',
 'unh.us.txt',
 'v.us.txt',
 'vz.us.txt',
 'wmt.us.txt']

In [2]:
bh_result_file = load_json_file("./results/buy_and_hold/bh_result")
bh_result_dict = load_json_file("./results/buy_and_hold/bh_result_table")
periods_file = load_json_file("./results/buy_and_hold/periods_dict")
# bh_result_file
# bh_result_dict
# periods_file

Buy and Hold Table

In [3]:
for time_format in time_formats:
    for exp in experiments:
        file_bh_minor = open(
            f"./results/buy_and_hold/{time_format}_{exp}_bh_results.csv",
            "w+",
            newline="",
        )
        exp_string = f"{exp}"
        with file_bh_minor:
            # identifying header
            header = [
                "Transactions",
                "Wins",
                "Losses",
                "Return",
                "Profit",
                "Loss",
                "Average Return",
                "Average Profit",
                "Average Loss",
            ]
            writer = csv.DictWriter(file_bh_minor, fieldnames=header)

            # writing data row-wise into the csv file
            writer.writeheader()
            writer.writerow(
                {
                    "Transactions": bh_result_dict[exp_string][time_format]["bets"],
                    "Wins": bh_result_dict[exp_string][time_format]["wins"],
                    "Losses": bh_result_dict[exp_string][time_format]["losses"],
                    "Return": round(
                        bh_result_dict[exp_string][time_format]["return"], 2
                    ),
                    "Profit": round(
                        bh_result_dict[exp_string][time_format]["profit"], 2
                    ),
                    "Loss": round(bh_result_dict[exp_string][time_format]["loss"], 2),
                    "Average Return": round(
                        bh_result_dict[exp_string][time_format][
                            "average_return_per_ticker"
                        ],
                        2,
                    ),
                    "Average Profit": round(
                        bh_result_dict[exp_string][time_format][
                            "average_profit_per_ticker"
                        ],
                        2,
                    ),
                    "Average Loss": round(
                        bh_result_dict[exp_string][time_format][
                            "average_loss_per_ticker"
                        ],
                        2,
                    ),
                }
            )

In [4]:
file_bh = open(f"./results/buy_and_hold/bh_results.csv", "w+", newline="")
with file_bh:
    # identifying header
    header = [
        "Experiment",
        "Time Format",
        "Transactions",
        "Wins",
        "Losses",
        "Return",
        "Profit",
        "Loss",
        "Average Return",
        "Average Profit",
        "Average Loss",
    ]
    writer = csv.DictWriter(file_bh, fieldnames=header)
    writer.writeheader()
    for exp in experiments:
        for time_format in time_formats:
            exp_string = f"{exp}"
            exp_csv_string = "Simulation" if exp == 20000 else "Normal"
            # writing data row-wise into the csv file
            writer.writerow(
                {
                    "Experiment": exp_csv_string,
                    "Time Format": time_format,
                    "Transactions": bh_result_dict[exp_string][time_format]["bets"],
                    "Wins": bh_result_dict[exp_string][time_format]["wins"],
                    "Losses": bh_result_dict[exp_string][time_format]["losses"],
                    "Return": round(
                        bh_result_dict[exp_string][time_format]["return"], 2
                    ),
                    "Profit": round(
                        bh_result_dict[exp_string][time_format]["profit"], 2
                    ),
                    "Loss": round(bh_result_dict[exp_string][time_format]["loss"], 2),
                    "Average Return": round(
                        bh_result_dict[exp_string][time_format][
                            "average_return_per_ticker"
                        ],
                        2,
                    ),
                    "Average Profit": round(
                        bh_result_dict[exp_string][time_format][
                            "average_profit_per_ticker"
                        ],
                        2,
                    ),
                    "Average Loss": round(
                        bh_result_dict[exp_string][time_format][
                            "average_loss_per_ticker"
                        ],
                        2,
                    ),
                }
            )

In [5]:
buy_and_hold_table = pd.read_csv("./results/buy_and_hold/bh_results.csv")
buy_and_hold_table

,Experiment,Time Format,Transactions,Wins,Losses,Return,Profit,Loss,Average Return,Average Profit,Average Loss
0,Normal,1_day,384,232,152,5683.84,11084.89,-5401.05,177.62,346.40,-168.78
1,Normal,1_hour,3775,2290,1485,23763.33,48028.24,-24264.91,742.60,1500.88,-758.28
2,Normal,5_minute,7840,4616,3224,31635.10,74325.21,-42690.12,988.60,2322.66,-1334.07
3,Simulation,1_day,384,232,152,963501.10,1479858.30,-516357.20,30109.41,46245.57,-16136.16
4,Simulation,1_hour,3775,2290,1485,3069730.11,5510479.05,-2440748.94,95929.07,172202.47,-76273.40
5,Simulation,5_minute,7840,4616,3224,4196507.18,8566792.67,-4370285.49,131140.85,267712.27,-136571.42


Periods Table

In [6]:
periods_dataframe_dict = dict()
for time_format in time_formats:
    optim_periods = []
    valid_periods = []

    # for i in range(len(periods_file['0'][time_format]['test'])):
    for i in range(18):
        if i % 2 == 0:
            optim_periods.append(periods_file["0"][time_format]["test"][i])
    optim_periods.append("...")
    optim_periods.append("...")
    optim_periods.append(periods_file["0"][time_format]["test"][-2])
    optim_periods.append(periods_file["0"][time_format]["test"][-4])
    optim_periods.append(periods_file["0"][time_format]["test"][-6])

    # for i in range(len(periods_file['0'][time_format]['valid'])):
    for i in range(18):
        if i % 2 == 0:
            valid_periods.append(periods_file["0"][time_format]["valid"][i])
    valid_periods.append("...")
    valid_periods.append("...")
    valid_periods.append(periods_file["0"][time_format]["valid"][-2])
    valid_periods.append(periods_file["0"][time_format]["valid"][-4])
    valid_periods.append(periods_file["0"][time_format]["valid"][-6])

    periods_dataframe_dict[f"{time_format} optimization"] = optim_periods
    periods_dataframe_dict[f"{time_format} validation"] = valid_periods

In [7]:
periods_dataframe = pd.DataFrame(periods_dataframe_dict)
periods_dataframe.to_csv(
    "./results/buy_and_hold/periods_dataframe_horizontal.csv", index=None
)
periods_dataframe

,1_day optimization,1_day validation,1_hour optimization,1_hour validation,5_minute optimization,5_minute validation
0,2020-03-26:2021-03-26,2021-03-26:2022-03-26,2020-02-11:2020-05-11,2020-05-11:2020-08-11,2020-02-07:2020-04-03,2020-04-03:2020-05-29
1,2020-06-26:2021-06-26,2021-06-26:2022-06-26,2020-02-25:2020-05-25,2020-05-25:2020-08-25,2020-02-14:2020-04-10,2020-04-10:2020-06-05
2,2020-09-26:2021-09-26,2021-09-26:2022-09-26,2020-03-10:2020-06-08,2020-06-08:2020-09-08,2020-02-21:2020-04-17,2020-04-17:2020-06-12
3,2020-12-26:2021-12-26,2021-12-26:2022-12-26,2020-03-24:2020-06-22,2020-06-22:2020-09-22,2020-02-28:2020-04-24,2020-04-24:2020-06-19
4,2021-03-26:2022-03-26,2022-03-26:2023-03-26,2020-04-07:2020-07-06,2020-07-06:2020-10-06,2020-03-06:2020-05-01,2020-05-01:2020-06-26
5,2021-06-26:2022-06-26,2022-06-26:2023-06-26,2020-04-21:2020-07-20,2020-07-20:2020-10-20,2020-03-13:2020-05-08,2020-05-08:2020-07-03
6,2021-09-26:2022-09-26,2022-09-26:2023-09-26,2020-05-05:2020-08-03,2020-08-03:2020-11-03,2020-03-20:2020-05-15,2020-05-15:2020-07-10
7,2021-12-26:2022-12-26,2022-12-26:2023-12-26,2020-05-19:2020-08-17,2020-08-17:2020-11-17,2020-03-27:2020-05-22,2020-05-22:2020-07-17
8,2022-03-26:2023-03-26,2023-03-26:2024-03-26,2020-06-02:2020-08-31,2020-08-31:2020-12-01,2020-04-03:2020-05-29,2020-05-29:2020-07-24
9,...,...,...,...,...,...


In [8]:
periods_dataframe_dict = dict()
optim_periods = []
valid_periods = []
for time_format in time_formats:
    optim_periods.append(time_format)
    valid_periods.append(time_format)
    # for i in range(len(periods_file['0'][time_format]['test'])):
    for i in range(18):
        if i % 2 == 0:
            optim_periods.append(periods_file["0"][time_format]["test"][i])
    optim_periods.append("...")
    optim_periods.append("...")
    optim_periods.append(periods_file["0"][time_format]["test"][-2])
    optim_periods.append(periods_file["0"][time_format]["test"][-4])
    optim_periods.append(periods_file["0"][time_format]["test"][-6])

    # for i in range(len(periods_file['0'][time_format]['valid'])):
    for i in range(18):
        if i % 2 == 0:
            valid_periods.append(periods_file["0"][time_format]["valid"][i])
    valid_periods.append("...")
    valid_periods.append("...")
    valid_periods.append(periods_file["0"][time_format]["valid"][-2])
    valid_periods.append(periods_file["0"][time_format]["valid"][-4])
    valid_periods.append(periods_file["0"][time_format]["valid"][-6])

periods_dataframe_dict[f"Optimization"] = optim_periods
periods_dataframe_dict[f"Validation"] = valid_periods

In [9]:
periods_dataframe = pd.DataFrame(periods_dataframe_dict)
periods_dataframe.to_csv(
    "./results/buy_and_hold/periods_dataframe_vertical.csv", index=None
)
periods_dataframe

,Optimization,Validation
0,1_day,1_day
1,2020-03-26:2021-03-26,2021-03-26:2022-03-26
2,2020-06-26:2021-06-26,2021-06-26:2022-06-26
3,2020-09-26:2021-09-26,2021-09-26:2022-09-26
4,2020-12-26:2021-12-26,2021-12-26:2022-12-26
5,2021-03-26:2022-03-26,2022-03-26:2023-03-26
6,2021-06-26:2022-06-26,2022-06-26:2023-06-26
7,2021-09-26:2022-09-26,2022-09-26:2023-09-26
8,2021-12-26:2022-12-26,2022-12-26:2023-12-26
9,2022-03-26:2023-03-26,2023-03-26:2024-03-26
